# M05. Leverage
- This predicts pitcher leverage to determine which types of pitchers to put into the game
- Type: Model
- Run Frequency: Irregular
- Sources:
    - MLB Stats API
    - Steamer
- Created: 12/30/2024
- Updated: 10/31/2025

### Imports

In [1]:
%run "C:\Users\james\Documents\MLB\Code\U01. Imports.ipynb"
%run "C:\Users\james\Documents\MLB\Code\U02. Functions.ipynb"
%run "C:\Users\james\Documents\MLB\Code\U03. Classes.ipynb"
%run "C:\Users\james\Documents\MLB\Code\U04. Datasets.ipynb"

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


### Data

##### Plate Appearances 

In [ ]:
complete_dataset = pd.read_csv(os.path.join(baseball_path, "PA Dataset.csv"))

##### Bullpens

In [4]:
def append_files_from_bullpens_folders(base_path):
    # Initialize an empty list to store dataframes
    dataframes = []

    # Walk through the directory and its subdirectories
    for root, dirs, files in os.walk(base_path):
        # Filter folders that start with "Bullpens 2024"
        for dir_name in dirs:
            if dir_name.startswith("Bullpens 2024") or dir_name.startswith("Bullpens 2025"):
                folder_path = os.path.join(root, dir_name)
                
                # Get all files in this folder
                for file in os.listdir(folder_path):
                    file_path = os.path.join(folder_path, file)

                    # Attempt to read the file into a dataframe
                    try:
                        df = pd.read_csv(file_path)  # Use the correct reader for your file format
                        dataframes.append(df)
                    except Exception as e:
                        print(f"Error reading {file_path}: {e}")

    # Concatenate all dataframes into a single dataframe
    if dataframes:
        combined_df = pd.concat(dataframes, ignore_index=True)
    else:
        combined_df = pd.DataFrame()  # Return an empty dataframe if no files found

        
    return combined_df

In [5]:
bullpen_df = append_files_from_bullpens_folders(r"C:\Users\james\Documents\MLB\Database\A04. Bullpens")

### Clean

Shrink dataset

In [6]:
complete_dataset = complete_dataset[['date', 'pitcher', 'pitcherName', 'halfInning', 'inning', 'prePitcherScore', 'preBatterScore', 'startingPitcher']]

### Merge

In [7]:
merged_df = complete_dataset.merge(bullpen_df, left_on=['pitcherName', 'date'], right_on=['Name', 'date'], how='inner', suffixes=("_Actual", "_Assigned"))

### Sample

Keep recent data

In [8]:
merged_df = merged_df[merged_df['date'].astype(str) > '20240101']

Only keep non-missing reliever leverages

In [9]:
merged_df = merged_df[merged_df['Leverage'] != 0]

### Model

$ \hat{\text{Leverage}} = pitcherLead + top + inning\_dummy\_list $

##### Inputs

Pitcher's team is leading

In [10]:
merged_df['pitcherLead'] = merged_df['prePitcherScore'] - merged_df['preBatterScore']

Top of the inning

In [11]:
merged_df['top'] = (merged_df['halfInning'] == "top").astype(int)

Inning dummies

In [12]:
for i in range(1, 12):
    merged_df[f'inning_{i}'] = (merged_df['inning'] == i).astype(int)
    
merged_df['inning_11'] = (merged_df['inning'] >= 11).astype(int)

In [13]:
inning_dummy_list = [col for col in merged_df.columns if col.startswith("inning_")]

Model inputs

In [26]:
leverage_input_list = ['pitcherLead', 'top'] + inning_dummy_list

##### Train/Test Split

Define features and target

In [15]:
X = merged_df[leverage_input_list]
y = merged_df['Leverage']

Split

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [17]:
X_train = torch.tensor(X_train.to_numpy(), dtype=torch.float32, device=device)
y_train = torch.tensor(y_train.to_numpy(), dtype=torch.long, device=device)
X_test  = torch.tensor(X_test.to_numpy(), dtype=torch.float32, device=device)
y_test  = torch.tensor(y_test.to_numpy(), dtype=torch.long, device=device)

##### Settings

In [18]:
num_classifiers = 3 # Ensemble size
random_state = random.randint(10000,90000) 

leverage_model_parameters = {
    'hidden_layer_sizes': (64,32),
    'activation': 'relu',
    'max_iter': 100,
    'alpha': 0.00001,
    'learning_rate_init': 0.01, 
    'batch_size': 'auto',
    'random_state': random_state,
    'early_stopping': True,
    'tol': 0.00001,
    'n_iter_no_change': 20,
    'validation_fraction': 0.05
}

##### Train

In [19]:
# # Define model
# predict_leverage = MLPClassifier(hidden_layer_sizes=(100,100), max_iter=500, random_state=42)
# predict_leverage.fit(X_train, y_train)

# # Save model
# pickle.dump(predict_leverage, open(os.path.join(model_path, "M05. Leverage", f"predict_leverage_{todaysdate}.sav"), 'wb'))

In [21]:
merged_df['Leverage'].value_counts()

Leverage
2    39478
3    22811
4     6378
Name: count, dtype: int64

In [28]:
# ------------------------
# SETTINGS
# ------------------------

input_size = X_train.shape[1]
hidden_layers = leverage_model_parameters["hidden_layer_sizes"]
lr = leverage_model_parameters["learning_rate_init"]
num_epochs = leverage_model_parameters["max_iter"]
random_seed = random_state

# Leverage classes are 2, 3, 4 → need output_size >= 5
output_size = 5     # valid class indices: 0,1,2,3,4

# Save directory
save_dir = os.path.join(model_path, "M05. Leverage", todaysdate)
os.makedirs(save_dir, exist_ok=True)
model_filename = f"predict_leverage.sav"

# ------------------------
# ENSEMBLE TRAINING
# ------------------------
ensemble = []

for j in range(num_classifiers):
    print(f"Training classifier {j+1}/{num_classifiers}")

    seed = random_seed + 100*j
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)

    # --- build model ---
    model = nn.Sequential(
        nn.Linear(input_size, hidden_layers[0]),
        nn.ReLU(),
        nn.Linear(hidden_layers[0], hidden_layers[1]),
        nn.ReLU(),
        nn.Linear(hidden_layers[1], output_size)   # must be 5
    ).to(device)

    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    # --- train ---
    model.train()
    for epoch in range(num_epochs):
        optimizer.zero_grad()
        outputs = model(X_train)       # logits
        loss = criterion(outputs, y_train)  # y_train contains 2,3,4
        loss.backward()
        optimizer.step()

    ensemble.append(model)

# ------------------------
# CONVERT TO NUMPY
# ------------------------
ensemble_numpy = []

for m in ensemble:
    sd = m.state_dict()
    layers = []

    linear_keys = [k for k in sd.keys() if "weight" in k]
    linear_keys.sort()

    for key in linear_keys:
        W = sd[key].cpu().numpy().T
        b = sd[key.replace("weight", "bias")].cpu().numpy()
        layers.append(W)
        layers.append(b)

    ensemble_numpy.append(layers)

# ------------------------
# BUILD NUMPY WRAPPER
# ------------------------
metadata = {
    "hidden_layers": hidden_layers,
    "num_classifiers": num_classifiers,
    "random_seed": random_seed,
    "training_epochs": num_epochs,
    "classes": [0,1,2,3,4]    # valid output indices
}

predict_leverage = NumpyPredict(
    ensemble_numpy=ensemble_numpy,
    input_columns=leverage_input_list,
    classes=[0,1,2,3,4],
    metadata=metadata
)

# ------------------------
# SAVE
# ------------------------
pickle_path = os.path.join(save_dir, model_filename)
with open(pickle_path, "wb") as f:
    pickle.dump(predict_leverage, f)

print(f"Saved leverage model → {pickle_path}")


Training classifier 1/3
Training classifier 2/3
Training classifier 3/3
Saved leverage model → C:\Users\james\Documents\MLB\Models\M05. Leverage\20251127\predict_leverage.sav


### Evaluate

This is a simple model with low stakes that performs quite well in the simulation, but evaluations should be created eventually to improve accuracy.